<a href="https://colab.research.google.com/github/Joanie-lab/dna-sequence-analysis/blob/main/dna_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DNA Sequence Analysis with Python

## What it does
This program reads DNA sequences from a FASTA file and calculates, for each sequence:
- Sequence length
- Counts of A, T, G and C
- GC content (percentage of bases that are G or C)
- Reverse complement (the sequence of the opposite DNA strand)

## How to use it
1. Run all cells in order (Runtime → Run all).
2. Upload your own FASTA file using the folder icon on the left, or create one as shown in Step 3.
3. Run `analyse_sequences("your_file_name.fasta")`.

## About the FASTA format

A FASTA file stores biological sequences as plain text. Each sequence starts with a header line beginning with `>`, followed by the sequence letters (A, T, G, C). Long sequences may be split across several lines.

In [1]:
fasta_text = """>seq1 Example sequence
ATGCGTACGTTAGC
>seq2 Another sequence
GGCTAATCGATCGA
"""

with open("sequences.fasta", "w") as f:
    f.write(fasta_text)

## Step 3: Create test data

In [2]:
with open("sequences.fasta") as f:
    print(f.read())

>seq1 Example sequence
ATGCGTACGTTAGC
>seq2 Another sequence
GGCTAATCGATCGA



## Step 4: Read the FASTA file

In [3]:
{"seq1": "ATGCGTACGTTAGC", "seq2": "GGCTAATCGATCGA"}

{'seq1': 'ATGCGTACGTTAGC', 'seq2': 'GGCTAATCGATCGA'}

In [4]:
def read_fasta(filename):
    sequences = {}
    name = None

    with open(filename) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                name = line[1:].split()[0]
                sequences[name] = ""
            elif name is not None:
                sequences[name] += line

    return sequences

In [5]:
data = read_fasta("sequences.fasta")
print(data)

{'seq1': 'ATGCGTACGTTAGC', 'seq2': 'GGCTAATCGATCGA'}


In [6]:
test_text = """>seq3 Longer example
ATGCGTACGTTAGC
GGCTAATCGATCGA
"""
with open("test_multiline.fasta", "w") as f:
    f.write(test_text)

print(read_fasta("test_multiline.fasta"))

{'seq3': 'ATGCGTACGTTAGCGGCTAATCGATCGA'}


## Step 5-7: Basic calculations
Functions to calculate sequence length, nucleotide counts and GC content.

In [7]:
def sequence_length(sequence):
    return len(sequence)

In [8]:
print(sequence_length(data["seq1"]))

14


In [9]:
def count_bases(sequence):
    sequence = sequence.upper()
    counts = {
        "A": sequence.count("A"),
        "T": sequence.count("T"),
        "G": sequence.count("G"),
        "C": sequence.count("C"),
    }
    return counts

In [10]:
counts = count_bases(data["seq1"])
print(counts)
print("Total counted:", sum(counts.values()))
print("Sequence length:", sequence_length(data["seq1"]))

{'A': 3, 'T': 4, 'G': 4, 'C': 3}
Total counted: 14
Sequence length: 14


GC content (%) = (G + C) / total length × 100

In [11]:
def gc_content(sequence):
    sequence = sequence.upper()
    g = sequence.count("G")
    c = sequence.count("C")
    return round((g + c) / len(sequence) * 100, 2)

In [12]:
print(gc_content("ATGC"))

50.0


In [13]:
print(gc_content(data["seq1"]))
print(gc_content(data["seq2"]))

50.0
50.0


## Step 8: Reverse complement
Creates the sequence of the opposite DNA strand by swapping each base for its partner (A↔T, G↔C) and reversing the result.

In [14]:
def reverse_complement(sequence):
    pairs = {"A": "T", "T": "A", "G": "C", "C": "G"}
    sequence = sequence.upper()

    complement = ""
    for base in sequence:
        complement += pairs[base]

    return complement[::-1]

In [15]:
print(reverse_complement("ATGC"))

GCAT


In [16]:
print(data["seq1"])
print(reverse_complement(data["seq1"]))

ATGCGTACGTTAGC
GCTAACGTACGCAT


In [17]:
original = data["seq1"]
twice = reverse_complement(reverse_complement(original))
print(original == twice)

True


## Step 9: Running the analysis
A function that reads a FASTA file and prints length, nucleotide counts, GC content and reverse complement for each sequence.


In [18]:
def analyse_sequences(filename):
    try:
        data = read_fasta(filename)
    except FileNotFoundError:
        print("Oops, I could not find the file:", filename)
        return

    for name, sequence in data.items():
        print("Name:", name)

        if len(sequence) == 0:
            print("This sequence is empty, skipping.")
            print()
            continue

        counts = count_bases(sequence)

        print("Length:", sequence_length(sequence))
        print("A:", counts["A"], " T:", counts["T"], " G:", counts["G"], " C:", counts["C"])
        print("GC content:", gc_content(sequence), "%")
        print("Reverse complement:", reverse_complement(sequence))
        print()

## Step 10: Testing on real data

Tested on NM_000207.3 (human insulin mRNA) downloaded from NCBI. The program read the 465-letter sequence and reported its nucleotide counts, a GC content of 63.87% and its reverse complement. The counts add up to the sequence length, which confirms the file was parsed correctly.

In [19]:
   real_text = """
>NM_000207.3 Homo sapiens insulin (INS), transcript variant 1, mRNA
AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCATGGCCCTGTG
GATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCTGACCCAGCCGCAGCCTTTGTGAAC
CAACACCTGTGCGGCTCACACCTGGTGGAAGCTCTCTACCTAGTGTGCGGGGAACGAGGCTTCTTCTACA
CACCCAAGACCCGCCGGGAGGCAGAGGACCTGCAGGTGGGGCAGGTGGAGCTGGGCGGGGGCCCTGGTGC
AGGCAGCCTGCAGCCCTTGGCCCTGGAGGGGTCCCTGCAGAAGCGTGGCATTGTGGAACAATGCTGTACC
AGCATCTGCTCCCTCTACCAGCTGGAGAACTACTGCAACTAGACGCAGCCCGCAGGCAGCCCCACACCCG
CCGCCTCCTGCACCGAGAGAGATGGAATAAAGCCCTTGAACCAGC
   """

   with open("real_sequence.fasta", "w") as f:
       f.write(real_text)

In [20]:
   analyse_sequences("real_sequence.fasta")

Name: NM_000207.3
Length: 465
A: 91  T: 77  G: 141  C: 156
GC content: 63.87 %
Reverse complement: GCTGGTTCAAGGGCTTTATTCCATCTCTCTCGGTGCAGGAGGCGGCGGGTGTGGGGCTGCCTGCGGGCTGCGTCTAGTTGCAGTAGTTCTCCAGCTGGTAGAGGGAGCAGATGCTGGTACAGCATTGTTCCACAATGCCACGCTTCTGCAGGGACCCCTCCAGGGCCAAGGGCTGCAGGCTGCCTGCACCAGGGCCCCCGCCCAGCTCCACCTGCCCCACCTGCAGGTCCTCTGCCTCCCGGCGGGTCTTGGGTGTGTAGAAGAAGCCTCGTTCCCCGCACACTAGGTAGAGAGCTTCCACCAGGTGTGAGCCGCACAGGTGTTGGTTCACAAAGGCTGCGGCTGGGTCAGGTCCCCAGAGGGCCAGCAGCGCCAGCAGGGGCAGGAGGCGCATCCACAGGGCCATGGCAGAAGGACAGTGATCTGCTTGATGGCCTCTTCTGATGCAGCCTGTCCTGGAGGGCT



## Step 11: Error handling

The program now handles three common problems without crashing: a missing file, empty sequences, and unexpected characters such as N. It prints a clear message instead and continues with the remaining sequences.

In [21]:
try:
    data = read_fasta("wrong_name.fasta")
except FileNotFoundError:
    print("Oops, I could not find that file.")

Oops, I could not find that file.


In [22]:
sequence = ""

if len(sequence) == 0:
    print("This sequence is empty, skipping.")
else:
    print(gc_content(sequence))

This sequence is empty, skipping.


In [23]:
sequence = "ATGC"

if len(sequence) == 0:
    print("This sequence is empty, skipping.")
else:
    print(gc_content(sequence))

50.0


In [24]:
def reverse_complement(sequence):
    pairs = {"A": "T", "T": "A", "G": "C", "C": "G"}
    sequence = sequence.upper()

    complement = ""
    for base in sequence:
            complement += pairs.get(base, "N")

    return complement[::-1]

In [25]:
print(reverse_complement("ATGN"))

NCAT


In [26]:
print(reverse_complement("ATGC"))

GCAT


In [27]:
analyse_sequences("real_sequence.fasta")

Name: NM_000207.3
Length: 465
A: 91  T: 77  G: 141  C: 156
GC content: 63.87 %
Reverse complement: GCTGGTTCAAGGGCTTTATTCCATCTCTCTCGGTGCAGGAGGCGGCGGGTGTGGGGCTGCCTGCGGGCTGCGTCTAGTTGCAGTAGTTCTCCAGCTGGTAGAGGGAGCAGATGCTGGTACAGCATTGTTCCACAATGCCACGCTTCTGCAGGGACCCCTCCAGGGCCAAGGGCTGCAGGCTGCCTGCACCAGGGCCCCCGCCCAGCTCCACCTGCCCCACCTGCAGGTCCTCTGCCTCCCGGCGGGTCTTGGGTGTGTAGAAGAAGCCTCGTTCCCCGCACACTAGGTAGAGAGCTTCCACCAGGTGTGAGCCGCACAGGTGTTGGTTCACAAAGGCTGCGGCTGGGTCAGGTCCCCAGAGGGCCAGCAGCGCCAGCAGGGGCAGGAGGCGCATCCACAGGGCCATGGCAGAAGGACAGTGATCTGCTTGATGGCCTCTTCTGATGCAGCCTGTCCTGGAGGGCT



In [28]:
analyse_sequences("not_a_real_file.fasta")

Oops, I could not find the file: not_a_real_file.fasta


In [29]:
with open("test_empty.fasta", "w") as f:
    f.write(">empty\n>ok\nATGC\n")

analyse_sequences("test_empty.fasta")

Name: empty
This sequence is empty, skipping.

Name: ok
Length: 4
A: 1  T: 1  G: 1  C: 1
GC content: 50.0 %
Reverse complement: GCAT



In [30]:
with open("test_n.fasta", "w") as f:
    f.write(">has_n\nATGNNCGA\n")

analyse_sequences("test_n.fasta")

Name: has_n
Length: 8
A: 2  T: 1  G: 2  C: 1
GC content: 37.5 %
Reverse complement: TCGNNCAT



## Example

Input (sequences.fasta):
```
>seq1 Example sequence
ATGCGTACGTTAGC
```

Output:
```
Name: seq1
Length: 14
A: 3  T: 4  G: 4  C: 3
GC content: 50.0 %
Reverse complement: GCTAACGTACGCAT
```

## Limitations
- Works with DNA only, not RNA or protein sequences.
- Unexpected characters such as N are reported as N in the reverse complement and are not included in the base counts.
- GC content divides by the full sequence length, including any N characters.
- Reads the whole file into memory, so it is not suited to very large genomes.
- Error handling is basic.

## How this could be extended
- Use Biopython (SeqIO) for more robust FASTA parsing.
- Translate DNA into protein sequences.
- Find motifs or open reading frames.
- Support RNA and ambiguous bases.
- Plot GC content along a sequence using matplotlib.
- Handle very large genomes efficiently.